# 🤖 Extractor de Criterios con Gemini

Este notebook procesa los archivos markdown de cada carrera y utiliza Gemini para extraer criterios estructurados.

## 🎯 Objetivo

Por cada perfil de carrera en formato markdown:
1. Cargar el prompt de extracción desde `prompt_criteria.md`
2. Enviar el contenido al modelo Gemini
3. Extraer criterios de INGRESO y EGRESO
4. Guardar en Excel `criterios.xlsx` con nuevas hojas

## 📋 Flujo

1. Cargar API key desde variables de entorno
2. Leer el prompt template
3. Iterar sobre los 66 archivos markdown
4. Procesar cada uno con Gemini
5. Agregar hojas `criterios_ingreso` y `criterios_egreso` al Excel

## 📦 Dependencias e Imports

In [7]:
import sys
import os
from pathlib import Path
import google.generativeai as genai
import json
from tqdm import tqdm
import time
import pandas as pd

from lib.set_envs import GEMINI_API_KEY

## ⚙️ Configuración de Gemini

In [ ]:
genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel('gemini-2.0-flash-exp')

print(f"✅ Gemini configurado correctamente")
print(f"🤖 Modelo: gemini-2.0-flash-exp")
print(f"🔑 API Key: {GEMINI_API_KEY[:10]}...")

## 📂 Configuración de Rutas

In [ ]:
PROMPT_PATH = Path("../assets/prompts/prompt_criteria.md")
MARKDOWN_DIR = Path("../assets/layouts/markdown")
EXCEL_PATH = Path("../assets/layouts/excel/criterios.xlsx")

print(f"📄 Prompt: {PROMPT_PATH}")
print(f"📁 Markdown dir: {MARKDOWN_DIR}")
print(f"📊 Excel output: {EXCEL_PATH}")

## 📖 Cargar Prompt Template

In [ ]:
with open(PROMPT_PATH, 'r', encoding='utf-8') as f:
    prompt_template = f.read()

print(f"✅ Prompt cargado ({len(prompt_template)} caracteres)")
print(f"\n--- Preview del prompt ---")
print(prompt_template[:300] + "..." if len(prompt_template) > 300 else prompt_template)

## 🔍 Listar Archivos Markdown

In [ ]:
markdown_files = sorted(MARKDOWN_DIR.glob("*.md"), key=lambda x: int(x.stem) if x.stem.isdigit() else 0)

print(f"📊 Total de archivos markdown: {len(markdown_files)}")
print(f"\nPrimeros 5 archivos:")
for f in markdown_files[:5]:
    print(f"  - {f.name}")

## 🤖 Función de Procesamiento

In [ ]:
def process_career_markdown(file_path: Path, prompt_template: str, model) -> dict:
    with open(file_path, 'r', encoding='utf-8') as f:
        markdown_content = f.read()
    
    full_prompt = f"{prompt_template}\n\n---\n\n{markdown_content}"
    
    try:
        response = model.generate_content(
            full_prompt,
            generation_config=genai.GenerationConfig(
                response_mime_type="application/json"
            )
        )
        
        data = json.loads(response.text)
        
        return {
            "career_id": int(file_path.stem) if file_path.stem.isdigit() else None,
            "status": "success",
            "criterios_ingreso": data.get("criterios_ingreso", []),
            "criterios_egreso": data.get("criterios_egreso", []),
            "error": None
        }
    except Exception as e:
        return {
            "career_id": int(file_path.stem) if file_path.stem.isdigit() else None,
            "status": "error",
            "criterios_ingreso": [],
            "criterios_egreso": [],
            "error": str(e)
        }

print("✅ Función de procesamiento definida")

## 🚀 Procesamiento en Lote

Este proceso puede tardar varios minutos dependiendo de la cantidad de archivos y la velocidad de la API.

In [ ]:
results = []

print(f"🚀 Iniciando procesamiento de {len(markdown_files)} archivos...\n")

for file_path in tqdm(markdown_files, desc="Procesando carreras"):
    result = process_career_markdown(file_path, prompt_template, model)
    results.append(result)
    
    time.sleep(1)

success_count = sum(1 for r in results if r["status"] == "success")
error_count = sum(1 for r in results if r["status"] == "error")

print(f"\n✅ Procesamiento completado")
print(f"   Exitosos: {success_count}")
print(f"   Errores: {error_count}")

## 📊 Estructurar Datos para Excel

In [ ]:
criterios_ingreso_list = []
for result in results:
    if result["status"] == "success":
        carrera_id = result["career_id"]
        for criterio in result["criterios_ingreso"]:
            criterios_ingreso_list.append({
                "carrera_id": carrera_id,
                "type_id": 1,  # INGRESO
                "riasec_id": criterio.get("riasec_id", 0),
                "desc_criterio": criterio.get("descripcion", ""),
                "resumen_criterio": criterio.get("resumen", "")
            })

criterios_egreso_list = []
for result in results:
    if result["status"] == "success":
        carrera_id = result["career_id"]
        for criterio in result["criterios_egreso"]:
            criterios_egreso_list.append({
                "carrera_id": carrera_id,
                "type_id": 2,  # EGRESO
                "riasec_id": criterio.get("riasec_id", 0),
                "desc_criterio": criterio.get("descripcion", ""),
                "resumen_criterio": criterio.get("resumen", "")
            })

df_ingreso = pd.DataFrame(criterios_ingreso_list)
df_egreso = pd.DataFrame(criterios_egreso_list)

print(f"\n📊 Datos estructurados:")
print(f"   Criterios de ingreso: {len(df_ingreso)} registros")
print(f"   Criterios de egreso: {len(df_egreso)} registros")

## 💾 Actualizar Excel

In [ ]:
excel_data = pd.read_excel(EXCEL_PATH, sheet_name=None)

print(f"📖 Hojas existentes: {list(excel_data.keys())}")

excel_data['criterios_ingreso'] = df_ingreso
excel_data['criterios_egreso'] = df_egreso

with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
    for sheet_name, df in excel_data.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"\n✅ Excel actualizado: {EXCEL_PATH}")
print(f"📄 Nuevas hojas agregadas:")
print(f"   - criterios_ingreso ({len(df_ingreso)} registros)")
print(f"   - criterios_egreso ({len(df_egreso)} registros)")

## 📊 Visualizar Muestra de Resultados

In [ ]:
print("\n=== MUESTRA CRITERIOS DE INGRESO ===")
print(df_ingreso.head(3))

print("\n=== MUESTRA CRITERIOS DE EGRESO ===")
print(df_egreso.head(3))

# Mostrar errores si hay
if error_count > 0:
    print("\n⚠️  ERRORES ENCONTRADOS:")
    for r in results:
        if r["status"] == "error":
            print(f"   Carrera ID {r['career_id']}: {r['error']}")